# WebDev RL · 评测（阶段 0 基线 / 训练前后对比）

用 **MiMo-V2.6-Distill-Qwen-9B** 对官方 webdev 题目单轮生成网页，截图，用视觉判官按官方组内奖励打分。

- **不填 `ADAPTER_DIR`**：阶段 0 基线。每题生成 8 份，看组内分数差异够不够 GRPO 学习。
- **填了 `ADAPTER_DIR`**：训练前后对比。每题基座生成 4 份、训练后生成 4 份，放进同一组让判官挑，直接比较谁更好。

运行时：**A100 GPU + 高 RAM**。判官 key 放在 Colab 密钥 `JUDGE_API_KEY`（或 VM 上的 `/content/.judge_key`）。
判官提示词和奖励公式移植自 [XiaomiMiMo/verl](https://github.com/XiaomiMiMo/verl) `mimo-oss` 分支（Apache-2.0）。

In [ ]:
import gc, json, os, shutil, subprocess, sys, time
from pathlib import Path

ON_VM = os.path.isdir("/content")
DRY_RUN_HTML_DIR = os.environ.get("DRY_RUN_HTML_DIR")  # 本地测试用：跳过 GPU 生成


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-3000:])
    return r.returncode


if not DRY_RUN_HTML_DIR:
    import torch
    assert torch.cuda.is_available(), "没有 GPU：运行时请选 A100"
    mem = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(torch.cuda.get_device_name(0), f"{mem:.0f} GiB")
    assert mem > 70, "需要 80GB 显存：选 A100 并打开「高 RAM」"

## 安装依赖
版本都选的是发布超过两周、互相兼容的：vLLM 0.30.0 支持 Qwen3.5 架构，且在 TRL 1.14.2 支持的范围内（训练 notebook 用同一套）。

In [ ]:
PINS = ["vllm==0.30.0", "transformers==5.18.0", "peft==0.21.2", "trl==1.14.2", "accelerate==1.15.0",
        "flash-linear-attention", "openai", "pandas", "pyarrow", "pillow", "playwright", "huggingface_hub", "datasets"]
if ON_VM and not DRY_RUN_HTML_DIR:
    try:
        import vllm
        assert vllm.__version__ == "0.30.0"
        print("依赖已安装")
    except Exception:
        sh(f"{sys.executable} -m pip install -q " + " ".join(f"'{p}'" for p in PINS))
        sh("playwright install --with-deps chromium")
        print("安装完成。如果后面 import 报错，重启会话后从下一格继续。")

## 配置

In [ ]:
MODEL = "XiaomiMiMo/MiMo-V2.6-Distill-Qwen-9B"
ADAPTER_DIR = os.environ.get("ADAPTER_DIR") or None   # 训练后的 LoRA 目录；留空 = 只评基座

N_TASKS = int(os.environ.get("N_TASKS", 16))   # 从留出的 100 道验证题里取前 N 道
N_SAMPLES = 8                                  # 每组 8 份；对比模式下基座 4 + 训练后 4
MAX_MODEL_LEN = 16384
MAX_NEW_TOKENS = 12288
SAMPLING = dict(temperature=0.6, top_p=0.95, top_k=20)  # 官方 webdev 配置

JUDGE = dict(judge_base_url=os.environ.get("JUDGE_BASE_URL", "https://openoneapi.com/v1"),
             judge_model=os.environ.get("JUDGE_MODEL", "deepseek-v4.1-flash"), judge_concurrency=8)

WORKDIR = Path(os.environ.get("WORKDIR") or ("/content/webdev_eval" if ON_VM else "webdev_eval_out"))
WORKDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 共享库：渲染、判官、奖励（两个 notebook 共用同一份代码）
LIB_FILES = {
    'webdev_common.py': '"""WebDev RL reward pipeline shared by the eval (stage 0) and training (stage 1) notebooks.\n\nSingle-turn generation of one HTML file, rendered locally with Playwright over http, gated on\nruntime errors, and scored with the official query judge + group pick, combined with the\nofficial group_v1 reward. Judge prompts and reward math are ported from XiaomiMiMo/verl\n(branch mimo-oss, recipes/design/grader_service, Apache-2.0).\n"""\n\nimport asyncio\nimport base64\nimport io\nimport json\nimport os\nimport random\nimport re\nimport shutil\nimport socket\nimport statistics as st\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nfrom PIL import Image\n\nImage.MAX_IMAGE_PIXELS = None\n\n# ---------------------------------------------------------------- settings\nCFG = dict(\n    judge_base_url="https://openoneapi.com/v1",\n    judge_model="deepseek-v4.1-flash",\n    judge_temperature=1.0,      # both official judges use 1.0\n    judge_concurrency=8,\n    pick_rounds=8,              # official: 8 rounds, Williams order\n    shot_w=1280, shot_h=800,    # screenshot viewport\n    shot_max_h=12000,           # full-page cap (official 12000)\n    judge_img_w=720,            # images are scaled to this width for the judges (official 720)\n    render_concurrency=6,\n    chromium_path=os.environ.get("CHROMIUM_PATH"),\n    mock_judge=os.environ.get("MOCK_JUDGE") == "1",\n)\nUSAGE = {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "failures": 0}\n_usage_lock = threading.Lock()\n\n\ndef configure(**kw):\n    unknown = set(kw) - set(CFG)\n    if unknown:\n        raise KeyError(f"unknown settings: {sorted(unknown)}")\n    CFG.update(kw)\n\n\ndef judge_api_key():\n    """Env var, then a key file (used when the VM is driven from the CLI), then Colab secrets."""\n    if os.environ.get("JUDGE_API_KEY"):\n        return os.environ["JUDGE_API_KEY"]\n    for p in ("/content/.judge_key", os.path.expanduser("~/.judge_key")):\n        if os.path.exists(p):\n            return Path(p).read_text().strip()\n    try:\n        from google.colab import userdata\n        return userdata.get("JUDGE_API_KEY")\n    except Exception:\n        return None\n\n\n# ---------------------------------------------------------------- tasks & prompts\ndef load_tasks():\n    """All 2093 official webdev tasks; the first 100 after a fixed shuffle are the held-out eval set."""\n    import pandas as pd\n    from huggingface_hub import hf_hub_download\n\n    df = pd.read_parquet(hf_hub_download("XiaomiMiMo/MiMo-V2.6-RL-oss", "webdev.parquet", repo_type="dataset"))\n    tasks = []\n    for _, row in df.iterrows():\n        inst = json.loads(row["extra_info"]["instance_json"])\n        tasks.append({"id": inst["instance_id"], "query": inst["problem_statement"].strip()})\n    random.Random(0).shuffle(tasks)\n    return tasks[:100], tasks[100:]\n\n\nSYSTEM_PROMPT = """You are an expert front-end engineer and web designer.\n\nBuild the website the user asks for and deliver it as ONE complete, self-contained HTML file.\nFor any request, always produce a working website; just build it, don\'t ask the user to confirm details.\n\nOutput rules:\n- Reply with exactly one fenced code block: ```html ... ``` containing the whole file. No other files exist.\n- Start the file with <!DOCTYPE html>.\n- Styling: Tailwind CSS via CDN — <script src="https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4"></script> — and/or an inline <style> tag.\n- JavaScript, if any, goes in inline <script> tags. If the request asks for React, load React, ReactDOM and Babel standalone from a CDN inside the same file.\n- Images may be referenced by absolute https URLs. Use mock content only; no backend.\n- Write plain HTML/CSS/JavaScript with normal angle brackets, not HTML entities.\n\nMake it complete, functional and visually polished."""\n\n\ndef build_messages(query):\n    return [\n        {"role": "system", "content": SYSTEM_PROMPT},\n        {"role": "user", "content": f"Build a website for the following request:\\n\\n{query}"},\n    ]\n\n\ndef extract_html(text):\n    """Pull the delivered HTML out of a model reply; None means no delivery."""\n    if not text:\n        return None\n    if "</think>" in text:\n        text = text.rsplit("</think>", 1)[1]\n    blocks = [m.group(2) for m in re.finditer(r"```([\\w+-]*)[ \\t]*\\n(.*?)```", text, flags=re.S)\n              if m.group(1).lower() in ("", "html")]\n    blocks = [b for b in blocks if re.search(r"<(html|body|!doctype)", b, flags=re.I)]\n    if blocks:\n        return max(blocks, key=len).strip()\n    m = re.search(r"(<!doctype html.*?</html>|<html.*?</html>)", text, flags=re.S | re.I)\n    return m.group(1).strip() if m else None\n\n\n# ---------------------------------------------------------------- rendering + runtime gate\nENV_NOISE = re.compile(r"Failed to fetch|NetworkError|net::|Load failed|ERR_|CORS", re.I)\nSKIP_SCRIPT_TYPES = {"text/babel", "application/json", "application/ld+json", "importmap", "text/template",\n                     "text/x-template", "text/plain", "text/markdown", "x-shader/x-vertex", "x-shader/x-fragment"}\nNODE = shutil.which("node") or ("/tools/node/bin/node" if os.path.exists("/tools/node/bin/node") else None)\n\n\ndef syntax_error(html, scratch):\n    """`node --check` every inline script; the first failure message, or None."""\n    if not NODE:\n        return None\n    for attrs, body in re.findall(r"<script(?![^>]*\\bsrc=)([^>]*)>(.*?)</script>", html, flags=re.S | re.I):\n        m = re.search(r"type\\s*=\\s*[\'\\"]([^\'\\"]+)", attrs, flags=re.I)\n        typ = m.group(1).lower() if m else ""\n        if typ in SKIP_SCRIPT_TYPES or not body.strip():\n            continue\n        p = Path(scratch) / ("_check.mjs" if typ == "module" else "_check.js")\n        p.write_text(body)\n        r = subprocess.run([NODE, "--check", str(p)], capture_output=True, text=True, timeout=15)\n        if r.returncode != 0:\n            lines = r.stderr.strip().splitlines()\n            hit = [l for l in lines if "Error" in l]\n            return (hit or lines or ["syntax error"])[0][:200]\n    return None\n\n\ndef _free_port():\n    s = socket.socket()\n    s.bind(("127.0.0.1", 0))\n    p = s.getsockname()[1]\n    s.close()\n    return p\n\n\nSCROLL_JS = """async (maxH) => {\n  const h = Math.min(document.documentElement.scrollHeight, maxH);\n  for (let y = 0; y < h; y += 600) { window.scrollTo(0, y); await new Promise(r => setTimeout(r, 120)); }\n  window.scrollTo(0, 0);\n}"""\n\n\nasync def _shoot_one(browser, url, out_path, sem):\n    res = {}\n    async with sem:\n        ctx = await browser.new_context(viewport={"width": CFG["shot_w"], "height": CFG["shot_h"]})\n        page = await ctx.new_page()\n        errors = []\n        page.on("pageerror", lambda e: errors.append(str(e)))\n        try:\n            async def work():\n                try:\n                    await page.goto(url, wait_until="networkidle", timeout=30000)\n                except Exception:\n                    pass  # a slow CDN is not the page\'s fault; shoot anyway\n                await page.wait_for_timeout(1200)\n                await page.evaluate(SCROLL_JS, CFG["shot_max_h"])\n                await page.wait_for_timeout(600)\n                return await page.screenshot(full_page=True, type="jpeg", quality=80)\n            jpg = await asyncio.wait_for(work(), timeout=75)\n            im = Image.open(io.BytesIO(jpg)).convert("RGB")\n            if im.height > CFG["shot_max_h"]:\n                im = im.crop((0, 0, im.width, CFG["shot_max_h"]))\n            im.save(out_path, "JPEG", quality=80)\n            res["shot"] = str(out_path)\n            res["shot_h"] = im.height\n            g = list(im.convert("L").resize((64, max(1, im.height * 64 // im.width))).tobytes())\n            mean = sum(g) / len(g)\n            res["blank"] = (sum((p - mean) ** 2 for p in g) / len(g)) ** 0.5 < 3\n        except asyncio.TimeoutError:\n            res["runtime_fail"] = "hang"\n        except Exception as e:\n            res["render_fail"] = str(e)[:200]\n        finally:\n            real = [e for e in errors if not ENV_NOISE.search(e)]\n            if real and "runtime_fail" not in res:\n                res["runtime_fail"] = "pageerror: " + real[0][:200]\n            await ctx.close()\n    return res\n\n\nasync def _shoot_all(jobs):\n    from playwright.async_api import async_playwright\n\n    async with async_playwright() as p:\n        exe = CFG["chromium_path"]\n        browser = await p.chromium.launch(**({"executable_path": exe} if exe else {}))\n        sem = asyncio.Semaphore(CFG["render_concurrency"])\n        out = await asyncio.gather(*(_shoot_one(browser, u, o, sem) for u, o in jobs))\n        await browser.close()\n    return out\n\n\ndef _run_async(coro):\n    """Run a coroutine whether or not an event loop is already running (Jupyter)."""\n    try:\n        asyncio.get_running_loop()\n    except RuntimeError:\n        return asyncio.run(coro)\n    box = {}\n    t = threading.Thread(target=lambda: box.setdefault("r", asyncio.run(coro)))\n    t.start()\n    t.join()\n    return box["r"]\n\n\ndef render(records, sites_dir):\n    """Write each record\'s HTML under sites_dir/<task>/<k>/, check scripts, screenshot over http."""\n    sites_dir = Path(sites_dir)\n    todo = []\n    for r in records:\n        if not r.get("html"):\n            continue\n        d = sites_dir / f"{r[\'task\']:04d}" / str(r["k"])\n        d.mkdir(parents=True, exist_ok=True)\n        (d / "index.html").write_text(r["html"])\n        r["dir"] = str(d)\n        err = syntax_error(r["html"], d)\n        if err:\n            r["runtime_fail"] = "syntax: " + err\n        todo.append(r)\n    if not todo:\n        return\n    port = _free_port()\n    server = subprocess.Popen([sys.executable, "-m", "http.server", str(port), "--bind", "127.0.0.1"],\n                              cwd=sites_dir, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n    try:\n        time.sleep(1.0)\n        jobs = [(f"http://127.0.0.1:{port}/{Path(r[\'dir\']).relative_to(sites_dir).as_posix()}/index.html",\n                 Path(r["dir"]) / "shot.jpg") for r in todo]\n        for r, res in zip(todo, _run_async(_shoot_all(jobs))):\n            if "runtime_fail" in r:\n                res.pop("runtime_fail", None)\n            r.update(res)\n    finally:\n        server.terminate()\n\n\n# ---------------------------------------------------------------- judges\nQUERY_FIT_PROMPT = \'# 维度：需求实现度 (query) —— 页面是否实现了 query 的要求\\n\\n> 对照用户 query 原文，看截图可见内容有没有把 query 要求的**主题、板块、要素、功能界面**做出来。\\n> 只评"做了没做、做对没做对"，不评好不好看（好不好看由其它维度负责）。\\n\\n## 任务\\n\\n你是网页需求验收员。下面给你一段用户 query 和该 query 对应产出的网页全页截图，判断截图可见内容对 query 的实现程度，打一个 0 到 1 的连续分数。\\n\\n## 判分依据\\n\\n先把 query 拆成若干**可见的要求点**：主题/行业、指定的板块（如首页 hero、产品列表、定价表、FAQ、联系表单…）、指定的要素（如某个具体数据、图表、地图、时间轴…）、指定的功能界面（如搜索框、筛选器、购物车、登录框、日历…）、指定的风格或语言。然后逐点对照截图勾选。\\n\\n分数区间：\\n- [0.0, 0.2)：截图可见内容基本没有实现 query，或主题完全错误\\n- [0.2, 0.4)：只实现少量表面元素，多数核心要求缺失\\n- [0.4, 0.6)：实现了主要方向，但多个关键板块、要素或功能界面明显缺失\\n- [0.6, 0.8)：主体要求已实现，仅有少量次要缺失或偏差\\n- [0.8, 1.0]：截图可见范围内完整、准确地实现了 query 的主题、板块、要素和功能界面\\n\\n先确定所在区间，再按缺失项的数量与重要性在区间内连续取值，不要只输出端点。\\n\\n## 注意\\n\\n- 只依据截图**可见**证据。截图看不到的交互（点击后才出现的内容）不算实现，也不算缺失，按可见部分判。\\n- query 没提的东西做了不加分也不扣分。\\n- 主题错了（做成了另一个行业/另一个产品）直接落到最低区间。\\n- 页面空白、渲染失败、只有骨架 → 最低区间。\\n- 不要因为页面漂亮就抬分，也不要因为页面丑就压分——这里只看需求覆盖。\\n\\n## 输出格式\\n\\n仅输出 JSON，不要 markdown：\\n{"reason": "一句话：query 要求了哪几点，截图实现了哪些、缺了哪些", "query_score": 0.0}\\n\'\n\nGROUP_PICK_PROMPT = \'# 组内挑选（group pick）—— 挑出明显更好的与明显更差的，其余不排\\n\\n> 一次调用看同一 query 的全部 rollout 截图（编号随机）。任务不是排全序，而是：\\n> 挑出**明显优于其他**的若干张（0 到多张）、**明显差于其他**的若干张（0 到多张）；\\n> 其余视为"一般"。如果整组看不出明显差别，就明确说"无明显差别"，**不要硬排**。\\n\\n## 任务\\n\\n下面是 {n} 个针对同一个建站需求生成的网页全页截图，编号 0 到 {n_max}（顺序随机，与质量无关）。\\n\\n请你以**整体审美观感**为主要依据，回答三个问题：\\n1. 哪几张**明显比组内其他的更好**？（可以是 0 张、1 张或多张）\\n2. 哪几张**明显比组内其他的更差**？（可以是 0 张、1 张或多张）\\n3. 如果全组水平接近、挑不出明显好坏，就直接判定"无明显差别"。\\n\\n建站需求（仅用于理解这是个什么页面、给视觉判断提供上下文，**不评需求实现度**）：\\n{query}\\n\\n## 怎么判"明显"\\n\\n- "明显"的标准是：**一个普通人把这几张并排看一眼，不用细想就能指出来**。需要反复对比、放大细看才能分出高下的，不算明显。\\n- 好坏都是**相对组内**而言，不看绝对水平：\\n  - 即使这一组单独看都没有质量特别高的，但有一张明显比其他的要好一些，也可以把它选进 good。\\n  - 反之，这一组质量都挺好，但有一两张稍逊一筹、且明显能看出来的，也可以把它们选进 bad。\\n- 宁可少选、不要多选。选出的每一张都应该能用一句话说清"为什么它明显比其他的好 / 差"。\\n- **不要为了凑数把中游的选进 good 或 bad**。\\n\\n## 整体审美观感看什么\\n\\n凭整体直觉：这个页面像不像一个成熟设计师做出来的成品？是否有统一的视觉语言、舒服的配色、清晰的层级、精致的素材、恰当的留白？看完第一反应是"这做得真好看"，还是"能用但普通"，还是"这很糟"。\\n\\n不要逐项打勾算分。但在组内比较时，下面这些**客观上公认的问题**是重要的参考：同组里带这些问题的页面，相对没有这些问题的页面更可能落到"明显差"。\\n\\n**渲染 / 结构类**\\n- 白屏、纯色空白页、只有骨架 / 灰框 / loading 状态、内容没渲染出来\\n- 页面大面积截断：图片、卡片、mockup、图表的边缘被切掉，尤其深色背景上的截断\\n- 板块之间夹着大段无内容的纯色空带；某个区块一半有内容、一半空着\\n- 元素互相重叠、文字压在图上看不清、内容溢出容器、明显错位\\n\\n**素材类**\\n- 用简陋 / 画崩 / 比例失调的 SVG 涂鸦冒充图片（人物五官走形、物体变形、剪贴画感）\\n- 占位符图片：灰框、破图、多张一模一样的占位 SVG 重复排列\\n- 把通用图标（放大镜、星星、箭头）放得极大当主视觉，空洞廉价\\n- 整页几乎没有任何视觉素材，只有文字 + 细线框，像文档站 / 线框稿\\n\\n**文字 / 配色类**\\n- 文字不可读：对比度过低、字压在复杂背景上、字号失控\\n- 配色刺眼冲突，或通篇单调死板（大片纯黑 / 纯白 / 纯灰，没有任何色彩层次）\\n\\n"明显好"则完全是主观感受：如果这一组里有哪一个页面相比其他的**明显让你眼前一亮**——特别有艺术感、设计感，或者特别酷炫——那它就是"明显好"。没有这种感觉，就不要选。\\n\\n## 注意\\n\\n- 编号是匿名候选，不要推测来源。\\n- 不判断内容是否正确、功能是否可用、需求是否实现——那些由其他评审负责。\\n- 多张页面如果几乎一样（同一模板、只是细节不同），它们应该同进同出，不要拆开。\\n\\n## 输出格式\\n\\n仅输出 JSON，不要 markdown：\\n{\\n  "reason": "先一句话说全组整体水平，再逐一说明每张被选进 good / bad 的理由（≤ 150 字）",\\n  "good": [明显更好的编号列表，可为空],\\n  "bad": [明显更差的编号列表，可为空],\\n  "no_clear_difference": false\\n}\\n\\n约束：good 与 bad 不能有交集；no_clear_difference 为 true 时 good 和 bad 必须都为空；不必覆盖所有编号，未列出的即为"一般"。\\n\'\n\n\ndef json_object(text):\n    """First balanced JSON object in text (brace matching, like the official grader)."""\n    s = text or ""\n    start = s.find("{")\n    while start >= 0:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(s)):\n            c = s[i]\n            if in_str:\n                if esc:\n                    esc = False\n                elif c == "\\\\":\n                    esc = True\n                elif c == \'"\':\n                    in_str = False\n                continue\n            if c == \'"\':\n                in_str = True\n            elif c == "{":\n                depth += 1\n            elif c == "}":\n                depth -= 1\n                if depth == 0:\n                    try:\n                        d = json.loads(s[start:i + 1])\n                        return d if isinstance(d, dict) else None\n                    except json.JSONDecodeError:\n                        break\n        start = s.find("{", start + 1)\n    return None\n\n\ndef img_b64(path, width=None):\n    width = width or CFG["judge_img_w"]\n    im = Image.open(path).convert("RGB")\n    if im.width > width:\n        im = im.resize((width, int(im.height * width / im.width)), Image.LANCZOS)\n    buf = io.BytesIO()\n    im.save(buf, "JPEG", quality=85)\n    return base64.b64encode(buf.getvalue()).decode()\n\n\ndef _image_part(b64):\n    return {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}\n\n\n_client = None\n\n\ndef _chat(parts, retries=3):\n    global _client\n    if _client is None:\n        from openai import OpenAI\n        _client = OpenAI(base_url=CFG["judge_base_url"], api_key=judge_api_key(), timeout=240)\n    content = [{"type": "text", "text": p} if isinstance(p, str) else p for p in parts]\n    last = None\n    for a in range(retries):\n        try:\n            resp = _client.chat.completions.create(model=CFG["judge_model"], temperature=CFG["judge_temperature"],\n                                                   messages=[{"role": "user", "content": content}])\n            with _usage_lock:\n                USAGE["calls"] += 1\n                if resp.usage:\n                    USAGE["prompt_tokens"] += resp.usage.prompt_tokens or 0\n                    USAGE["completion_tokens"] += resp.usage.completion_tokens or 0\n            return resp.choices[0].message.content or ""\n        except Exception as e:\n            last = e\n            time.sleep(3 * (a + 1))\n    with _usage_lock:\n        USAGE["failures"] += 1\n    raise RuntimeError(f"judge call failed: {last}")\n\n\ndef judge_query(query, shot_b64):\n    if CFG["mock_judge"]:\n        return {"query_score": round(random.Random(hash(shot_b64)).uniform(0.1, 1.0), 2), "reason": "mock"}\n    for _ in range(3):\n        try:\n            d = json_object(_chat([f"【用户 query】\\n{query}\\n\\n【该 query 的产出页面截图】：", _image_part(shot_b64), QUERY_FIT_PROMPT]))\n        except RuntimeError:\n            return None\n        if d is not None:\n            try:\n                s = float(d.get("query_score"))\n                if 0.0 <= s <= 1.0:\n                    return {"query_score": s, "reason": str(d.get("reason", ""))[:400]}\n            except (TypeError, ValueError):\n                pass\n    return None\n\n\ndef williams_orders(n, seed=0):\n    """Even n: n orders, each shot in each position once, each ordered adjacency once."""\n    base, lo, hi = [0], 1, n - 1\n    while len(base) < n:\n        base.append(hi)\n        hi -= 1\n        if len(base) < n:\n            base.append(lo)\n            lo += 1\n    perm = list(range(n))\n    random.Random(seed).shuffle(perm)\n    return [[perm[(x + r) % n] for x in base] for r in range(n)]\n\n\ndef pick_orders(n, rounds, seed=0):\n    if rounds == n and n % 2 == 0:\n        return williams_orders(n, seed)\n    rnd = random.Random(seed)\n    return [rnd.sample(range(n), n) for _ in range(rounds)]\n\n\ndef pick_once(shots_b64, query, order):\n    n = len(shots_b64)\n    if CFG["mock_judge"]:\n        rnd = random.Random(sum(order) + len(shots_b64[order[0]]))\n        good = [order[0]] if rnd.random() < 0.7 else []\n        bad = [order[-1]] if rnd.random() < 0.7 and order[-1] not in good else []\n        return {"good": good, "bad": bad, "no_clear_difference": not (good or bad), "order": order}\n    rubric = GROUP_PICK_PROMPT.replace("{n}", str(n)).replace("{n_max}", str(n - 1)).replace("{query}", query)\n    parts = []\n    for k, idx in enumerate(order):\n        parts += [f"【编号 {k}】", _image_part(shots_b64[idx])]\n    parts.append(rubric)\n    try:\n        d = json_object(_chat(parts))\n    except RuntimeError:\n        d = None\n    if d is None:\n        return {"error": True, "order": order}\n\n    def ints(lst):\n        out = []\n        for x in lst if isinstance(lst, list) else []:\n            try:\n                out.append(int(x))\n            except (TypeError, ValueError):\n                pass\n        return out\n    good = sorted({order[k] for k in ints(d.get("good")) if 0 <= k < n})\n    bad = [i for i in sorted({order[k] for k in ints(d.get("bad")) if 0 <= k < n}) if i not in good]\n    ncd = bool(d.get("no_clear_difference")) and not (good or bad)\n    return {"good": good, "bad": bad, "no_clear_difference": ncd, "reason": str(d.get("reason", ""))[:400], "order": order}\n\n\n# ---------------------------------------------------------------- reward (official group_v1)\nPICK_MIN_OK, PICK_NCD_ZERO_MIN, PICK_NET_DEADZONE = 5, 2, 1\nQUERY_DEDUCT = [(0.9, 0.0), (0.6, 0.2), (0.4, 0.4), (0.2, 0.8)]\nRAW_LO, RAW_HI = -2.0, 1.0\n\n\ndef combine(votes, n):\n    """Official vote combination (group_pick.py: combine)."""\n    R = len(votes)\n    ok = R >= PICK_MIN_OK\n    if not R:\n        return {"ok": ok, "pick_norm": [0.0] * n, "rounds": 0, "n_ncd": 0}\n    n_ncd = sum(1 for v in votes if v[0] == "N")\n    ncd = PICK_NCD_ZERO_MIN > 0 and n_ncd >= PICK_NCD_ZERO_MIN\n    pick_sum = [sum(1 if v[i] == "G" else -1 if v[i] == "B" else 0 for v in votes) for i in range(n)]\n    noise = [i for i in range(n) if pick_sum[i] != 0 and abs(pick_sum[i]) <= PICK_NET_DEADZONE]\n    eff = [0 if (ncd or i in noise) else pick_sum[i] for i in range(n)]\n    return {"ok": ok, "pick_norm": [s / R for s in eff] if ok else [0.0] * n, "rounds": R, "n_ncd": n_ncd}\n\n\ndef reward_one(pick_norm, query_score, runtime_ok):\n    """Official reward_one: (pick_norm - deduct + 2) / 3, with query and runtime floors."""\n    if query_score is None or pick_norm is None:\n        return None\n    if not runtime_ok:\n        return 0.0\n    for lo, d in QUERY_DEDUCT:\n        if query_score >= lo:\n            return round((pick_norm - d - RAW_LO) / (RAW_HI - RAW_LO), 4)\n    return 0.0\n\n\ndef grade(records, queries):\n    """Fill query_score / pick_norm / reward / status on records.\n\n    records: dicts with task (index into queries), k, html, and render output.\n    Group = all records sharing a task index. Mirrors the official rules:\n    no delivery -> 0, runtime failure -> 0, render/judge failure -> None (invalid).\n    """\n    shot_rows = [r for r in records if "shot" in r]\n    with ThreadPoolExecutor(CFG["judge_concurrency"]) as pool:\n        for r, b in zip(shot_rows, pool.map(lambda r: img_b64(r["shot"]), shot_rows)):\n            r["b64"] = b\n        for r, q in zip(shot_rows, pool.map(lambda r: judge_query(queries[r["task"]], r["b64"]), shot_rows)):\n            r["query_score"] = q["query_score"] if q else None\n            r["query_reason"] = q["reason"] if q else None\n\n        groups = {}\n        for r in shot_rows:\n            groups.setdefault(r["task"], []).append(r)\n        calls = []\n        for ti, rows in groups.items():\n            if len(rows) < 2:\n                continue\n            for o in pick_orders(len(rows), CFG["pick_rounds"], seed=ti):\n                calls.append((ti, o))\n        results = list(pool.map(lambda c: pick_once([r["b64"] for r in groups[c[0]]], queries[c[0]], c[1]), calls))\n\n    rounds_by_task = {}\n    for (ti, _), res in zip(calls, results):\n        rounds_by_task.setdefault(ti, []).append(res)\n    for ti, rows in groups.items():\n        votes = []\n        for rd in rounds_by_task.get(ti, []):\n            if rd.get("error"):\n                continue\n            if rd["no_clear_difference"]:\n                votes.append(["N"] * len(rows))\n            else:\n                votes.append(["G" if i in rd["good"] else "B" if i in rd["bad"] else "-" for i in range(len(rows))])\n        pn = combine(votes, len(rows))["pick_norm"] if len(rows) >= 2 else [0.0] * len(rows)\n        for r, p in zip(rows, pn):\n            r["pick_norm"] = p\n\n    for r in records:\n        if not r.get("html"):\n            r["reward"], r["status"] = 0.0, "no_delivery"\n        elif r.get("render_fail"):\n            r["reward"], r["status"] = None, "render_fail"\n        elif r.get("runtime_fail") and "shot" not in r:\n            r["reward"], r["status"] = 0.0, "runtime_fail"\n        else:\n            rw = reward_one(r.get("pick_norm"), r.get("query_score"), not r.get("runtime_fail"))\n            r["reward"] = rw\n            r["status"] = ("runtime_fail" if r.get("runtime_fail") else "ok") if rw is not None else "judge_fail"\n        r.pop("b64", None)\n    return records\n\n\ndef fill_invalid_with_group_mean(records):\n    """Official convention: an invalid row gets its group\'s mean valid reward, so it adds no gradient."""\n    by_task = {}\n    for r in records:\n        by_task.setdefault(r["task"], []).append(r)\n    out = []\n    for r in records:\n        if r["reward"] is not None:\n            out.append(r["reward"])\n        else:\n            vals = [x["reward"] for x in by_task[r["task"]] if x["reward"] is not None]\n            out.append(sum(vals) / len(vals) if vals else 0.0)\n    return out\n\n\ndef summarize(records):\n    valid = [r for r in records if r["reward"] is not None]\n    by_task = {}\n    for r in valid:\n        by_task.setdefault(r["task"], []).append(r["reward"])\n    stds = {t: (st.pstdev(v) if len(v) > 1 else 0.0) for t, v in by_task.items()}\n    status = {}\n    for r in records:\n        status[r["status"]] = status.get(r["status"], 0) + 1\n    qv = [r["query_score"] for r in records if r.get("query_score") is not None]\n    return {\n        "n": len(records),\n        "status": status,\n        "truncated": sum(r.get("finish") == "length" for r in records),\n        "mean_reward": round(st.mean(r["reward"] for r in valid), 4) if valid else None,\n        "mean_query_score": round(st.mean(qv), 4) if qv else None,\n        "mean_group_std": round(st.mean(stds.values()), 4) if stds else None,\n        "flat_groups": sum(s < 0.05 for s in stds.values()),\n        "groups": len(stds),\n        "mean_gen_tokens": round(st.mean(r.get("gen_tokens", 0) for r in records)) if records else 0,\n        "judge_usage": dict(USAGE),\n    }\n',
    'lora_sync.py': '"""Drift-free LoRA weight sync from a PEFT model into a colocated vLLM engine.\n\nTRL (1.14.x) syncs a PEFT model by calling ``merge_adapter()`` and then ``unmerge_adapter()``\non the live bf16 base weights every step. Each round trip rounds, so the frozen base weights\nslowly drift. This computes ``W + scaling * B @ A`` out of place in fp32 and casts once,\nleaving the trained model untouched.\n"""\n\n\ndef iter_merged_params(self):\n    """Drop-in for ``VLLMGeneration._iter_named_params`` on a single, unsharded PEFT model."""\n    from peft.tuners.lora import LoraLayer\n\n    model = self.model\n    lora_modules = {n: m for n, m in model.named_modules() if isinstance(m, LoraLayer)}\n    for name, param in model.named_parameters():\n        clean = name.removeprefix("base_model.model.").replace(".base_layer", "")\n        if model.prefix in clean or "original_module" in clean:\n            continue\n        data = param.data\n        if name.endswith(".base_layer.weight"):\n            module = lora_modules.get(name[: -len(".base_layer.weight")])\n            if module is not None:\n                delta = None\n                for a in module.active_adapters:\n                    if a in module.lora_A:\n                        d = (module.lora_B[a].weight.float() @ module.lora_A[a].weight.float()) * module.scaling[a]\n                        delta = d if delta is None else delta + d\n                if delta is not None:\n                    data = (data.float() + delta).to(data.dtype)\n        yield self._fix_param_name_to_vllm(clean, extra_prefixes=["modules_to_save.default."]), data\n\n\ndef install():\n    """Patch TRL so single-GPU PEFT runs use the drift-free sync; sharded runs keep TRL\'s own path."""\n    from trl.generation import vllm_generation as vg\n\n    original = vg.VLLMGeneration._iter_named_params\n\n    def patched(self):\n        if vg.is_peft_model(self.model) and not self._dist.is_fsdp:\n            yield from iter_merged_params(self)\n        else:\n            yield from original(self)\n\n    vg.VLLMGeneration._iter_named_params = patched\n',
}
LIB_DIR = WORKDIR / "lib"
LIB_DIR.mkdir(parents=True, exist_ok=True)
for _name, _src in LIB_FILES.items():
    (LIB_DIR / _name).write_text(_src)
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
import importlib
import webdev_common as wc
import lora_sync
importlib.reload(wc)
importlib.reload(lora_sync)

In [ ]:
wc.configure(**JUDGE)
if not wc.CFG["mock_judge"]:
    assert wc.judge_api_key(), "找不到判官 key：在 Colab 密钥里添加 JUDGE_API_KEY"
VAL, TRAIN = wc.load_tasks()
tasks = VAL[:N_TASKS]
print(f"验证集 {len(VAL)} 道，本次评测 {len(tasks)} 道；训练集 {len(TRAIN)} 道")

## 生成

In [ ]:
def generate(model_path, queries, n, seed=0):
    """vLLM 生成：每个 query n 份。返回 [(qi, text, n_tokens, finish_reason)]。"""
    from vllm import LLM, SamplingParams

    llm = LLM(model=model_path, max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.85, seed=seed,
              limit_mm_per_prompt={"image": 0, "video": 0})
    t0 = time.time()
    outs = llm.chat([wc.build_messages(q) for q in queries], SamplingParams(n=n, max_tokens=MAX_NEW_TOKENS, **SAMPLING))
    print(f"{model_path}: 生成 {len(queries)}×{n} 用时 {time.time() - t0:.0f}s")
    rows = [(qi, c.text, len(c.token_ids), c.finish_reason) for qi, o in enumerate(outs) for c in o.outputs]
    del llm
    gc.collect()
    import torch
    torch.cuda.empty_cache()
    return rows


def merge_adapter(base_id, adapter_dir, out_dir):
    """把 LoRA 并进基座另存一份，供 vLLM 加载。增量在 fp32 里算，只在最后转一次 bf16。"""
    import torch
    from huggingface_hub import snapshot_download
    from peft import PeftModel
    from peft.tuners.lora import LoraLayer
    from transformers import AutoModelForImageTextToText

    out_dir = Path(out_dir)
    if (out_dir / "model.safetensors.index.json").exists():
        return str(out_dir)
    snapshot_download(base_id, local_dir=out_dir, allow_patterns=["*.json", "*.jinja", "*.txt", "*.model", "*.tiktoken"])
    base = AutoModelForImageTextToText.from_pretrained(base_id, dtype=torch.bfloat16, device_map="cuda")
    model = PeftModel.from_pretrained(base, adapter_dir)
    with torch.no_grad():
        for m in model.modules():
            if isinstance(m, LoraLayer):
                for a in m.active_adapters:
                    if a in m.lora_A:
                        w = m.base_layer.weight
                        delta = (m.lora_B[a].weight.float() @ m.lora_A[a].weight.float()) * m.scaling[a]
                        w.data = (w.data.float() + delta).to(w.dtype)
    model.unload().save_pretrained(out_dir, safe_serialization=True)
    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    return str(out_dir)


queries = [t["query"] for t in tasks]
records = []
if DRY_RUN_HTML_DIR:
    fixtures = sorted(Path(DRY_RUN_HTML_DIR).glob("*.html"))
    for ti in range(len(tasks)):
        for k in range(N_SAMPLES):
            f = fixtures[(ti + k) % len(fixtures)]
            text = "```html\n" + f.read_text() + "\n```" if f.stem != "nodelivery" else "sorry"
            src = "adapter" if (ADAPTER_DIR and k >= N_SAMPLES // 2) else "base"
            records.append({"task": ti, "k": k, "source": src, "text": text, "gen_tokens": len(text) // 4, "finish": "stop"})
else:
    plan = [("base", MODEL, N_SAMPLES // 2 if ADAPTER_DIR else N_SAMPLES)]
    if ADAPTER_DIR:
        plan.append(("adapter", merge_adapter(MODEL, ADAPTER_DIR, WORKDIR / "merged"), N_SAMPLES // 2))
    offset = {}
    for src, path, n in plan:
        for qi, text, ntok, fin in generate(path, queries, n):
            k = offset.get(qi, 0)
            offset[qi] = k + 1
            records.append({"task": qi, "k": k, "source": src, "text": text, "gen_tokens": ntok, "finish": fin})

for r in records:
    r["html"] = wc.extract_html(r["text"])
print(f"{len(records)} 份：交付 {sum(bool(r['html']) for r in records)}，截断 {sum(r['finish'] == 'length' for r in records)}")

## 截图 + 判官 + 奖励

In [ ]:
sites = WORKDIR / "sites"
shutil.rmtree(sites, ignore_errors=True)
t0 = time.time()
wc.render(records, sites)
print(f"截图 {sum('shot' in r for r in records)}，运行检查失败 {sum('runtime_fail' in r for r in records)}，"
      f"渲染失败 {sum('render_fail' in r for r in records)}，用时 {time.time() - t0:.0f}s")
t0 = time.time()
wc.grade(records, queries)
print(f"判官用时 {time.time() - t0:.0f}s，调用 {wc.USAGE['calls']} 次，失败 {wc.USAGE['failures']} 次")

## 报告
组内标准差是关键：GRPO 只从同组差异里学习，一组 8 份分数几乎相同（标准差 < 0.05）就学不到东西。
对比模式下看 `head_to_head`：同组里训练后的样本被判官选为更好（pick 为正）的程度。

In [ ]:
import statistics as st

summary = wc.summarize(records)
summary.update(model=MODEL, adapter=ADAPTER_DIR, judge_model=wc.CFG["judge_model"], n_tasks=len(tasks))
if ADAPTER_DIR:
    for src in ("base", "adapter"):
        rs = [r for r in records if r["source"] == src]
        q = [r["query_score"] for r in rs if r.get("query_score") is not None]
        p = [r["pick_norm"] for r in rs if r.get("pick_norm") is not None]
        summary[f"{src}_query_score"] = round(st.mean(q), 4) if q else None
        summary[f"{src}_pick_norm"] = round(st.mean(p), 4) if p else None
        summary[f"{src}_no_delivery_or_runtime_fail"] = sum(r["status"] in ("no_delivery", "runtime_fail") for r in rs)
    summary["head_to_head"] = (None if summary["adapter_pick_norm"] is None or summary["base_pick_norm"] is None
                               else round(summary["adapter_pick_norm"] - summary["base_pick_norm"], 4))
print(json.dumps(summary, ensure_ascii=False, indent=2))
if not ADAPTER_DIR:
    ok = 1 - summary["flat_groups"] / max(1, summary["groups"])
    print("\n结论：", "组内差异足够，可以开始训练。" if ok >= 0.5 else "超过一半的组几乎没有差异，先排查判官或采样温度。")

rows = [{k: v for k, v in r.items() if k not in ("text", "html")} | {"query": queries[r["task"]]} for r in records]
(WORKDIR / "results.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=1))
print("已保存", WORKDIR / "results.json")

## 看图：每题最高分和最低分

In [ ]:
def gallery(n_tasks=4):
    html = []
    for ti in range(min(n_tasks, len(tasks))):
        rows = sorted([r for r in records if r["task"] == ti and r["reward"] is not None and "shot" in r], key=lambda r: r["reward"])
        if not rows:
            continue
        cells = []
        for tag, r in (("最高", rows[-1]), ("最低", rows[0])):
            b = wc.img_b64(r["shot"], 360)
            cells.append(f"<div style='flex:1;min-width:260px'><b>{tag} reward={r['reward']:.3f}</b> [{r['source']}] "
                         f"query={r.get('query_score')} pick={r.get('pick_norm', 0):+.2f}<br>"
                         f"<img src='data:image/jpeg;base64,{b}' style='width:100%;max-height:520px;object-fit:cover;object-position:top;border:1px solid #ccc'></div>")
        html.append(f"<h4>题 {ti}：{queries[ti][:120]}…</h4><div style='display:flex;gap:16px;flex-wrap:wrap'>{''.join(cells)}</div>")
    return "".join(html)


(WORKDIR / "gallery.html").write_text(gallery())
if "ipykernel" in sys.modules:
    from IPython.display import HTML, display
    display(HTML(gallery()))
else:
    print("画廊已保存", WORKDIR / "gallery.html")